# Duplicate Rx Agent Demo\n## 1. Problem framing\nRules-first medication reconciliation assist before eRx transmission. Clinician remains final decision-maker.

In [ ]:
import json\nfrom pathlib import Path\nimport sys\n\nROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebook' else Path.cwd()\nif str(ROOT) not in sys.path:\n    sys.path.append(str(ROOT))\n\nfrom src.models import PendingPrescriptionEvent, MedicationHistoryEntry\nfrom src.rules import apply_rules\nfrom src.runner import run_duplicate_check\n

## 2. Inputs

In [ ]:
data_dir = ROOT / 'data'\npending = json.loads((data_dir / 'pending_rx.json').read_text(encoding='utf-8'))\nhistory = json.loads((data_dir / 'med_history.json').read_text(encoding='utf-8'))\npending['event_time']

## 3. Deterministic overlap helpers\n## 4. Candidate filtering\n## 5. Rules engine

In [ ]:
event = PendingPrescriptionEvent.from_dict(pending)\nrows = [MedicationHistoryEntry.from_dict(x) for x in history]\nrules_result = apply_rules(event, rows)\nrules_result.pending_start_date, rules_result.pending_end_date, rules_result.max_overlap_days

## 6. Optional Claude call for ambiguity handling\nClaude is only used for ambiguous `transition_or_duplicate` candidates.

In [ ]:
print('Metformin ignored:', any('ingredient_mismatch' in r.ignore_reasons and 'metformin' in r.drug_display.lower() for r in rules_result.all_rows))\nprint('Same-strength semaglutide candidate count:', len(rules_result.true_duplicate_same_drug))\nif rules_result.true_duplicate_same_drug:\n    top = sorted(rules_result.true_duplicate_same_drug, key=lambda x: x.overlap_days, reverse=True)[0]\n    print('Top evidence:', top.drug_display, top.fill_date.isoformat(), 'overlap_days=', top.overlap_days)

## 7. Final finding JSON\n## 8. Result interpretation\n## 9. Limitations and next steps

In [ ]:
result = run_duplicate_check(pending, history)\nprint('severity =', result['finding']['severity'])\nprint('max_overlap_days =', result['finding']['computed']['max_overlap_days'])\nprint(json.dumps(result['finding'], indent=2))